# Property Integrals from PySCF for FermiCG

This notebook walks through the full integral generation pipeline:

1. **SCF** – ROHF or UHF with PySCF
2. **SPADE partitioning** – define active space and cluster assignment from AOs
3. **Hamiltonian integrals** – `h0`, `h1`, `h2` in the active MO basis (with embedding)
4. **Property integrals** – electric dipole (and optionally velocity/momentum)
   transformed to the same active MO basis
5. **Save** – everything needed for the Julia CMF + TPSCI pipeline

The two-step orbital transformation chain is:
```
AO basis  --[Cact]--> active MO basis  --[U from cmf_oo_newton]--> CMF-optimized basis
```
This notebook handles the first step (`Cact`). The second step (`U`) is
applied automatically in `cmf.jl` via `orbital_rotation(ints, U)`
and the analogous loop over dipole components added there.

## 0. Imports

In [15]:
import numpy as np
import scipy
import pyscf
import pyscf.tools
import pyscf.ao2mo
import pyscf.scf
import pyscf.gto

from orbitalpartitioning import *

## 1. Molecule and SCF

Set up the molecule and run ROHF. Adjust `molecule`, `basis`, `spin`, and
`charge` for your system. For a closed-shell singlet use `spin=0` and
`pyscf.scf.RHF`.

In [16]:
molecule = """
 Cr 0.82627800 -1.30446200 -0.96524300
 Cr -0.82625500  1.30449100  0.96521400
 O  0.00001100  0.00001500 -0.00001400
 N  2.71379100 -0.58517300 -0.32156700
 H  3.52017600 -0.87998300 -0.88101600
 H  2.98302700 -0.84252100  0.62848200
 H  2.75216700  0.43162900 -0.34666000
 N  0.90611400 -0.01191500 -2.64373800
 H  0.15767300  0.67336300 -2.59732700
 H  0.79520000 -0.45289000 -3.56150100
 H  1.76766300  0.52845600 -2.74461100
 N  0.74644200 -2.59700900  0.71325300
 H  0.86887400 -3.59499400  0.51496700
 H -0.15371900 -2.54555600  1.18538900
 H  1.44183900 -2.41670700  1.43797600
 N -1.06123500 -2.02375100 -1.60891800
 H -1.51982700 -2.67507800 -0.96956400
 H -1.07760500 -2.52098500 -2.50431600
 H -1.71705200 -1.25706600 -1.73635500
 N  1.78821400 -2.82312800 -2.08895700
 H  2.20721200 -2.51946500 -2.97231100
 H  1.19687500 -3.61071400 -2.36865500
 H  2.56853800 -3.28036700 -1.60828700
 N -2.71376800  0.58520300  0.32153900
 H -2.72742600 -0.43151600  0.28926300
 H -3.01495100  0.88961900 -0.60504900
 H -3.51124000  0.82996900  0.91689300
 N -0.90609100  0.01194400  2.64371000
 H -0.86251400  0.45862400  3.56440800
 H -0.11862500 -0.63055500  2.63771800
 H -1.73984000 -0.57546100  2.70589400
 N  1.06125800  2.02378000  1.60888900
 H  1.08580100  2.47872200  2.52630400
 H  1.50134100  2.71049000  0.99382500
 H  1.72819800  1.26071800  1.68834600
 N -0.74641900  2.59703800 -0.71328200
 H -0.92607700  3.58837700 -0.52509900
 H -1.40673500  2.37764300 -1.45939400
 H  0.17080800  2.59240800 -1.15542700
 N -1.78819100  2.82315700  2.08892800
 H -1.19783000  3.61531400  2.35756800
 H -2.19798000  2.52398300  2.97800300
 H -2.57540900  3.27269300  1.61233200
"""

basis  = "def2-svp"
spin   = 6     # number of unpaired electrons (2S)
charge = 4

pymol = pyscf.gto.Mole(
    atom     = molecule,
    symmetry = True,
    spin     = spin,
    charge   = charge,
    basis    = basis,
)
pymol.build()
print("symmetry:", pymol.topgroup)
print("n_AO    :", pymol.nao)

symmetry: C1
n_AO    : 366


In [17]:
mf = pyscf.scf.ROHF(pymol)
mf.verbose    = 4
mf.conv_tol   = 1e-8
mf.chkfile    = "scf.fchk"
mf.init_guess = "chkfile"
mf.run(max_cycle=200)

print("\nHartree-Fock energy: %12.8f Ha" % mf.e_tot)

S      = mf.get_ovlp()
F      = mf.get_fock()
C      = mf.mo_coeff          # alpha MOs (ROHF gives same spatial orbitals)
ndocc  = mf.nelec[1]          # doubly occupied
nsing  = mf.nelec[0] - ndocc  # singly occupied (open shell)
nvirt  = pymol.nao - ndocc - nsing

Cdocc = C[:, :ndocc]
Csing = C[:, ndocc:ndocc+nsing]
Cvirt = C[:, ndocc+nsing:]

print(f"ndocc={ndocc}  nsing={nsing}  nvirt={nvirt}")



******** <class 'pyscf.scf.rohf.ROHF'> ********
method = ROHF
initial guess = chkfile
damping factor = 0
level_shift factor = 0
DIIS = <class 'pyscf.scf.diis.CDIIS'>
diis_start_cycle = 1
diis_space = 8
diis_damp = 0
SCF conv_tol = 1e-08
SCF conv_tol_grad = None
SCF max_cycles = 200
direct_scf = True
direct_scf_tol = 1e-13
chkfile to save SCF result = scf.fchk
max_memory 4000 MB (current use 0 MB)
num. doubly occ = 73  num. singly occ = 6
Set gradient conv threshold to 0.0001
init E= -2721.66784100119
  HOMO = -0.639241187068676  LUMO = -0.388806543428867
cycle= 1 E= -2721.66784100119  delta_E= -1.82e-12  |g|= 8.23e-07  |ddm|= 5.54e-06
  HOMO = -0.639241141389168  LUMO = -0.388806545650786
Extra cycle  E= -2721.66784100117  delta_E= 1.73e-11  |g|= 8.58e-07  |ddm|= 6.83e-06
converged SCF energy = -2721.66784100117

Hartree-Fock energy: -2721.66784100 Ha
ndocc=73  nsing=6  nvirt=287


## 2. SPADE Orbital Partitioning

Define which AOs belong to each fragment, then use SVD-based subspace
projection (SPADE) to select the active MOs and assign them to clusters.

Edit the `frag1`, `frag2`, `frag3` lists and the shell labels (`'3d'`, `'2p'`, …)
to match your system.

In [18]:
# ---- Define fragment AOs by atom index and shell type ----
# ao_labels returns (atom_idx, atom_symbol, shell_type, magnetic_quantum_number)
frag1, frag2, frag3 = [], [], []
full = []

for ao_idx, ao in enumerate(mf.mol.ao_labels(fmt=False)):
    atom_idx, _, shell, _ = ao
    if atom_idx == 0 and shell in ["3d", "4d"]:    # Cr1
        frag1.append(ao_idx); full.append(ao_idx)
    elif atom_idx == 2 and shell in ["2p", "3p"]:  # bridging O
        frag2.append(ao_idx); full.append(ao_idx)
    elif atom_idx == 1 and shell in ["3d", "4d"]:  # Cr2
        frag3.append(ao_idx); full.append(ao_idx)

frags = [frag1, frag2, frag3]
print("fragment AO indices:", frags)

fragment AO indices: [[14, 15, 16, 17, 18, 19, 20, 21, 22, 23], [65, 66, 67, 68, 69, 70], [45, 46, 47, 48, 49, 50, 51, 52, 53, 54]]


In [19]:
# ---- Project MOs onto full active space ----
X    = scipy.linalg.sqrtm(S)          # symmetric square root of overlap
Pf   = [X[:, f] for f in frags]       # per-fragment projectors
Pfull = X[:, full]                    # projector onto union of all fragments

(Oact, Sact, Vact), (Cenv, Cerr, _) = svd_subspace_partitioning(
    (Cdocc, Csing, Cvirt), Pfull, S
)
assert Cerr.shape[1] == 0, "Unexpected error space"

print(f"Active: {Oact.shape[1]} occ + {Sact.shape[1]} singly + {Vact.shape[1]} virt")
print(f"Environment: {Cenv.shape[1]} doubly occupied")

 Partition  366 orbitals into a total of   26 orbitals
            Index   Sing. Val. Space       
                0   1.02099767            2*
                1   1.01920192            2*
                2   1.01230129            2*
                3   0.99139991            2*
                4   0.98172397            2*
                5   0.97853083            2*
                6   0.97056793            1*
                7   0.97046332            1*
                8   0.95191147            1*
                9   0.95183391            1*
               10   0.95045591            1*
               11   0.95027147            1*
               12   0.94507014            2*
               13   0.80017315            2*
               14   0.79720515            2*
               15   0.78597834            0*
               16   0.76414052            0*
               17   0.76400926            0*
               18   0.74105112            2*
               19   0.73777440            2*
 

In [20]:
# ---- Split active space into per-fragment clusters ----
init_fspace = []
clusters    = []
Cfrags      = []
orb_index   = 1   # 1-based for Julia

for fi, f in enumerate(frags):
    (Of, Sf, Vf), _ = svd_subspace_partitioning(
        (Oact, Sact, Vact), Pf[fi], S
    )
    Cfrags.append(np.hstack((Of, Sf, Vf)))
    ndocc_f = Of.shape[1]
    init_fspace.append((ndocc_f + Sf.shape[1], ndocc_f))
    nmof = Of.shape[1] + Sf.shape[1] + Vf.shape[1]
    clusters.append(list(range(orb_index, orb_index + nmof)))
    orb_index += nmof

# Symmetrically orthogonalize across fragments
Cfrags = sym_ortho(Cfrags, S)
Cact   = np.hstack(Cfrags)

print("init_fspace:", init_fspace)
print("clusters   :", clusters)
print("Cact shape :", Cact.shape)

 Partition   26 orbitals into a total of   10 orbitals
            Index   Sing. Val. Space       
                0   0.98007469            2*
                1   0.95989523            1*
                2   0.95973768            1*
                3   0.95782428            2*
                4   0.95185911            1*
                5   0.72128300            2*
                6   0.66266369            2*
                7   0.65977588            2*
                8   0.41649385            0*
                9   0.37877034            0*
               10   0.37632183            0
               11   0.34825176            0
               12   0.22437928            0
               13   0.05495206            0
               14   0.05419836            0
               15   0.04890146            1
               16   0.04865723            1
               17   0.04258182            2
               18   0.02320185            2
               19   0.02306068            2
           

## 3. Hamiltonian Integrals in Active MO Basis

Standard embedded active-space integrals with the doubly-occupied environment
treated via a mean-field embedding potential.

In [21]:
# ---- Embedding density from environment (doubly occupied) MOs ----
d1_embed = 2.0 * Cenv @ Cenv.T   # (n_AO, n_AO)

# Sanity check: no embedded electrons in active space
n_embed_in_act = np.trace(S @ d1_embed @ S @ Cact @ Cact.T)
print(f"Embedded electrons leaking into active space: {n_embed_in_act:.2e}  (should be ~0)")

h_core = pyscf.scf.hf.get_hcore(pymol)
j_emb, k_emb = pyscf.scf.hf.get_jk(pymol, d1_embed, hermi=1)

# Nuclear repulsion + environment contribution to h0
h0  = pyscf.gto.mole.energy_nuc(pymol)
h0 += np.trace(d1_embed @ (h_core + 0.5 * j_emb - 0.25 * k_emb))

# One-electron integrals in active MO basis
h_ao  = h_core + j_emb - 0.5 * k_emb          # effective one-electron Hamiltonian
h1    = Cact.T @ h_ao @ Cact                   # (n_act, n_act)

# Two-electron integrals in active MO basis
n_act = Cact.shape[1]
h2    = pyscf.ao2mo.kernel(pymol, Cact, aosym="s4", compact=False)
h2.shape = (n_act, n_act, n_act, n_act)

print(f"h0 = {h0:.8f}  |h1| = {np.linalg.norm(h1):.4f}  h2 shape = {h2.shape}")

Embedded electrons leaking into active space: -2.18e-16  (should be ~0)
h0 = -2642.76577527  |h1| = 30.8395  h2 shape = (26, 26, 26, 26)


## 4. Property Integrals in Active MO Basis

### 4.1 Electric Dipole — Length Gauge

The length-gauge dipole operator is
$$\hat{\mu}_\alpha = \sum_{pq} \langle p | r_\alpha | q \rangle \sum_\sigma \hat{p}^\dagger_{p\sigma} \hat{q}_\sigma$$

`int1e_r` gives $\langle \mu | r_x | \nu \rangle,\, \langle \mu | r_y | \nu \rangle,\, \langle \mu | r_z | \nu \rangle$
in the AO basis, shape `(3, n_AO, n_AO)`.

The transformation to the active MO basis is identical to `h1`:
$$\mu^\alpha_{pq} = \sum_{\mu\nu} C_{\mu p}\, \langle \mu | r_\alpha | \nu \rangle\, C_{\nu q}$$

In [22]:
# Dipole integrals in AO basis: shape (3, n_AO, n_AO), components x, y, z
dip_ao = pymol.intor('int1e_r', comp=3)

# Transform to active MO basis: dip_mo[x, p, q] = Cact.T @ dip_ao[x] @ Cact
# Using einsum: sum over AO indices mu, nu
dip_mo = np.einsum('mp, xmn, nq -> xpq', Cact, dip_ao, Cact)  # (3, n_act, n_act)

print("dip_mo shape:", dip_mo.shape)
print("  x-component norm:", np.linalg.norm(dip_mo[0]))
print("  y-component norm:", np.linalg.norm(dip_mo[1]))
print("  z-component norm:", np.linalg.norm(dip_mo[2]))

# Symmetry check: dipole integrals are real and symmetric
for x, lbl in enumerate('xyz'):
    asym = np.max(np.abs(dip_mo[x] - dip_mo[x].T))
    print(f"  max |dip_{lbl} - dip_{lbl}.T| = {asym:.2e}  (should be ~0)")

dip_mo shape: (3, 26, 26)
  x-component norm: 6.697771522297793
  y-component norm: 10.414066789880321
  z-component norm: 7.776824119086693
  max |dip_x - dip_x.T| = 2.44e-15  (should be ~0)
  max |dip_y - dip_y.T| = 5.44e-15  (should be ~0)
  max |dip_z - dip_z.T| = 4.97e-15  (should be ~0)


### 4.2 Electric Dipole — Velocity (Momentum) Gauge  *(optional)*

The velocity-gauge transition dipole uses the momentum operator
$\hat{p} = -i\nabla$:
$$f_{0n}^{\rm vel} = \frac{2}{3\,\Delta E_{0n}}
  \left|\langle 0 | \nabla | n \rangle\right|^2$$

`int1e_ipovlp` gives $\langle \mu | \nabla | \nu \rangle$ (purely imaginary,
antisymmetric). For exact wavefunctions the length and velocity oscillator
strengths are equal — this is a useful numerical check.

In [23]:
# Momentum (nabla) integrals in AO basis: shape (3, n_AO, n_AO)
# Note: these are purely imaginary in the real-AO basis (antisymmetric)
nabla_ao = pymol.intor('int1e_ipovlp', comp=3)  # <mu|d/dx|nu>

nabla_mo = np.einsum('mp, xmn, nq -> xpq', Cact, nabla_ao, Cact)  # (3, n_act, n_act)

print("nabla_mo shape:", nabla_mo.shape)
# Should be antisymmetric: nabla[x, p, q] = -nabla[x, q, p]
for x, lbl in enumerate('xyz'):
    asym = np.max(np.abs(nabla_mo[x] + nabla_mo[x].T))
    print(f"  max |nabla_{lbl} + nabla_{lbl}.T| = {asym:.2e}  (should be ~0 for antisymmetry)")

nabla_mo shape: (3, 26, 26)
  max |nabla_x + nabla_x.T| = 6.45e-16  (should be ~0 for antisymmetry)
  max |nabla_y + nabla_y.T| = 1.07e-15  (should be ~0 for antisymmetry)
  max |nabla_z + nabla_z.T| = 1.11e-15  (should be ~0 for antisymmetry)


### 4.3 Compute SCF-level dipole moment  *(sanity check)*

The SCF dipole using only the mean-field 1-RDM in the active space should
agree with PySCF's built-in `mf.dip_moment()` (within the active space
approximation).

In [24]:
# ---- Active-space 1-RDM from ROHF ----
# P_act[p,q] = Cact.T @ S @ P_total_AO @ S @ Cact
Pa_total, Pb_total = mf.make_rdm1()    # alpha and beta 1-RDMs in AO basis
P_total_AO = Pa_total + Pb_total

Pa_act = Cact.T @ S @ Pa_total @ S @ Cact
Pb_act = Cact.T @ S @ Pb_total @ S @ Cact
P_act  = Pa_act + Pb_act

# Electronic dipole in active space (length gauge)
dip_elec_act = np.array([
    -np.trace(dip_mo[x] @ P_act)   # negative: electrons have charge -e
    for x in range(3)
])

# Nuclear contribution (all atoms, not just active ones)
charges = pymol.atom_charges()
coords  = pymol.atom_coords()   # in Bohr
dip_nuc = np.einsum('i, ix -> x', charges, coords)

dip_total = dip_nuc + dip_elec_act
print("Dipole (active space, length gauge) [a.u.]:")
print(f"  x={dip_total[0]:.6f}  y={dip_total[1]:.6f}  z={dip_total[2]:.6f}")
print(f"  |mu| = {np.linalg.norm(dip_total):.6f} a.u.")

# PySCF reference (full system)
dip_pyscf = mf.dip_moment(verbose=0)
print("\nPySCF full-system dipole [Debye]:")
print(f"  x={dip_pyscf[0]:.6f}  y={dip_pyscf[1]:.6f}  z={dip_pyscf[2]:.6f}")

Dipole (active space, length gauge) [a.u.]:
  x=0.009266  y=-0.000217  z=-0.013112
  |mu| = 0.016058 a.u.

PySCF full-system dipole [Debye]:
  x=-0.000074  y=0.001427  z=0.000553


## 4.3 Spin-Orbit Coupling (SOC) Integrals

The one-electron Breit-Pauli spin-orbit Hamiltonian is

$$
\hat{H}^\text{BP}_\text{SOC}
= \frac{\alpha^2}{2}\sum_K Z_K
  \sum_{p,q,\sigma,\sigma'}
  \langle p | \frac{\mathbf{r}-\mathbf{R}_K}{|\mathbf{r}-\mathbf{R}_K|^3}
  \times \hat{\mathbf{p}} | q \rangle_{\sigma\sigma'}
  \hat{p}^\dagger_{p\sigma}\hat{q}_{\sigma'}
$$

In PySCF this is accessed via `int1e_spnucsp` (4-component: scalar + 3 Cartesian).
The SOMF (spin-orbit mean-field) approximation adds a screened two-electron
contribution; for heavy transition metals the one-center, one-electron
approximation is often sufficient.

The integrals are purely imaginary for real AO bases, so we store their
imaginary parts.  The SOC matrix element between TPSCI roots $r_1$ and $r_2$
is then

$$
H^\text{SOC}_{r_1 r_2}
= \sum_{pq}\Bigl[
    h^z_{pq}(\gamma^{r_1 r_2}_{\alpha\alpha,pq} - \gamma^{r_1 r_2}_{\beta\beta,pq})
  + h^+_{pq}\,\gamma^{r_1 r_2}_{\alpha\beta,pq}
  + h^-_{pq}\,\gamma^{r_1 r_2}_{\beta\alpha,pq}
  \Bigr]
$$

where $h^z, h^\pm$ are derived from the three Cartesian SOC tensors.


In [25]:
# ---- SOC integrals (Breit-Pauli nuclear, imaginary part) ----
# int1e_spnucsp returns shape (4, nao, nao):
#   comp 0: scalar (spin-free),  1: Lx,  2: Ly,  3: Lz
# These are antisymmetric (purely imaginary for real AO bases).
soc_ao = pymol.intor('int1e_spnucsp', comp=4)

# We only need the three Cartesian spin components (1,2,3)
# h_SOC[x, p, q] = Cact.T @ soc_ao[x+1] @ Cact   (x = 0,1,2 → Lx,Ly,Lz)
soc_mo = np.einsum('mp, xmn, nq -> xpq', Cact, soc_ao[1:], Cact)  # (3, n_act, n_act)

# Prefactor: alpha^2 / 2  (alpha = fine-structure constant ~ 1/137.036)
alpha_fs = 1.0 / 137.035999084
soc_mo *= (alpha_fs**2 / 2.0)

print("SOC integrals shape:", soc_mo.shape)
print("Max |soc_mo|: %.6e" % np.max(np.abs(soc_mo)))

# h^z = L_z component → spin-conserving (M_S unchanged)
# h^+ = L_x + i L_y  → spin-raising (alpha annihilate beta, ΔM_S = +1)
# h^- = L_x - i L_y  → spin-lowering (beta annihilate alpha, ΔM_S = -1)
# For a real wavefunction treatment, keep all three separately.
soc_Lx = soc_mo[0]   # (n_act, n_act)
soc_Ly = soc_mo[1]
soc_Lz = soc_mo[2]

# Also compute SOMF correction (two-electron mean-field screening):
# The SOMF adds -1/2 J_screen + K_screen (Boettger factors).
# Here we use the simpler one-electron nuclear + exchange-only screening.
# For production runs, use pyscf.mcscf.casscf or pyscf.prop.zfs.
try:
    from pyscf.prop import soc_jk
    somf = soc_jk.get_jk(mf, hermi=0)
    somf_mo = np.einsum('mp, xmn, nq -> xpq', Cact, somf[1:], Cact) * (alpha_fs**2 / 2.0)
    print("SOMF two-electron correction computed.")
    soc_Lx += somf_mo[0]
    soc_Ly += somf_mo[1]
    soc_Lz += somf_mo[2]
except ImportError:
    print("SOMF module not available; using one-electron SOC only.")


SOC integrals shape: (3, 26, 26)
Max |soc_mo|: 2.625340e-02
SOMF module not available; using one-electron SOC only.


## 4.4 Two-Electron Repulsion Integrals (ERI) in Active MO Basis

The full ERI tensor $(pq|rs)$ in the active MO basis is needed for:
- The 2-RDM energy expression $E_2 = \frac{1}{2}\sum_{pqrs}(pq|rs)\Gamma_{pq,rs}$
- Inter-cluster exchange coupling $J_{IJ} = \frac{1}{2}\sum_{p,r\in I,\,q,s\in J}(pr|qs)\Gamma_{pq,rs}$

The storage convention matches `ints_h2.npy` already saved (chemist's notation,
8-fold symmetry). We also extract the cluster-pair exchange integrals directly
for efficient $J$ computation.


In [26]:
# ---- Full ERI in active MO basis (already in ints_h2, re-expose for clarity) ----
# h2[p,q,r,s] = (pq|rs) = integral phi_p(1) phi_q(1) 1/r12 phi_r(2) phi_s(2)
# This was already computed and saved as ints_h2.npy; reload if needed:
# h2 = np.load('ints_h2.npy')

print("ERI tensor shape:", h2.shape)
print("Max |h2|: %.6e" % np.max(np.abs(h2)))

# ---- Cluster-pair exchange integrals (pr|qs) with p,r in I and q,s in J ----
# For a 3-cluster system with cluster orbital ranges I_orbs, J_orbs, K_orbs:
n_act = Cact.shape[1]
cluster_ranges = []
offset = 0
for cl in clusters:
    cluster_ranges.append(slice(offset, offset + len(cl)))
    offset += len(cl)

print("\nCluster orbital ranges:")
for i, sl in enumerate(cluster_ranges):
    print(f"  Cluster {i+1}: orbitals {sl.start}..{sl.stop-1}")

# Exchange integrals between cluster 1 and cluster 2: K[p,q,r,s] = (pr|qs)
# p,r in cluster 1 (I),  q,s in cluster 2 (J)
sl_I = cluster_ranges[0]
sl_J = cluster_ranges[1]
K_IJ = h2[sl_I, sl_I, sl_J, sl_J]    # shape (norb_I, norb_I, norb_J, norb_J)
# Note: h2[p,q,r,s]=(pq|rs); for exchange we want (pr|qs) = h2[p,r,q,s].
# Reorder:
K_IJ_exch = h2[sl_I, sl_J, sl_I, sl_J].transpose(0,2,1,3)   # (norb_I, norb_I, norb_J, norb_J)
#   i.e. K_IJ_exch[p,r,q,s] = (pr|qs) for p,r in I; q,s in J
print("\nExchange integral K_IJ shape:", K_IJ_exch.shape)
print("Max |K_IJ|: %.6e" % np.max(np.abs(K_IJ_exch)))
# Exchange integrals between cluster 1 and cluster 3: K[p,q,r,s] = (pr|qs)
sl_K = cluster_ranges[2]
K_IK = h2[sl_I, sl_I, sl_K, sl_K]    # shape (norb_I, norb_I, norb_K, norb_K)
K_IK_exch = h2[sl_I, sl_K, sl_I, sl_K].transpose(0,2,1,3)   # (norb_I, norb_I, norb_K, norb_K)
print("\nExchange integral K_IK shape:", K_IK_exch.shape)
print("Max |K_IK|: %.6e" % np.max(np.abs(K_IK_exch)))
# Exchange integrals between cluster 2 and cluster 3: K[p,q,r,s] = (pr|qs)
K_JK = h2[sl_J, sl_J, sl_K, sl_K]    # shape (norb_J, norb_J, norb_K, norb_K)
K_JK_exch = h2[sl_J, sl_K, sl_J, sl_K].transpose(0,2,1,3)   # (norb_J, norb_J, norb_K, norb_K)
print("\nExchange integral K_JK shape:", K_JK_exch.shape)
print("Max |K_JK|: %.6e" % np.max(np.abs(K_JK_exch)))
# Exchange integrals between cluster 2 and cluster 3: K[p,q,r,s] = (pr|qs)


ERI tensor shape: (26, 26, 26, 26)
Max |h2|: 8.690605e-01

Cluster orbital ranges:
  Cluster 1: orbitals 0..9
  Cluster 2: orbitals 10..15
  Cluster 3: orbitals 16..25

Exchange integral K_IJ shape: (10, 10, 6, 6)
Max |K_IJ|: 5.703338e-02

Exchange integral K_IK shape: (10, 10, 10, 10)
Max |K_IK|: 3.731224e-02

Exchange integral K_JK shape: (6, 6, 10, 10)
Max |K_JK|: 5.360962e-02


## 4.5 Orbital Centers and Spin-Dipolar (D_SS) Integrals

The spin-spin (magnetic dipolar) contribution to zero-field splitting is:

$$D^\text{SS}_{ab} = \frac{g_e^2\alpha^2}{4\,S(2S-1)}
  \sum_{p,q,r,s} T_{ab}[p,q,r,s]\,\Gamma^{\text{spin}}_{pqrs}$$

where $\Gamma^\text{spin} = \Gamma_{\alpha\alpha}+\Gamma_{\beta\beta}-\Gamma_{\alpha\beta}-\Gamma_{\beta\alpha}$.

**Orbital-center approximation** (point-dipole, suitable for well-separated clusters):

$$D^\text{SS}_{ab} \approx \frac{g_e^2\alpha^2}{4\,S(2S-1)}
  \sum_{p\in I,q\in J} K_{pq}\,t_{ab}(\mathbf{R}_p,\mathbf{R}_q)$$

where $K_{pq}=(pq|pq)$ and $\mathbf{R}_p=\langle\phi_p|\mathbf{r}|\phi_p\rangle$ (diagonal dipole).

We save `orb_centers_26.npy` and `T_cl_SS_26.npy` for Julia's `spin_correlators.jl`.

In [27]:
# ===================================================================
# 4.5  Orbital Centers and Classical Spin-Dipolar Tensor for D_SS (26-orbital)
# ===================================================================

# ---- Orbital centroids: diagonal dipole matrix elements ----
orb_centers_mo = np.array([dip_mo[x].diagonal() for x in range(3)]).T  # (n_act, 3)
print("Orbital centers in active MO basis (Bohr):")
print(f"  shape: {orb_centers_mo.shape}  (n_act={n_act}, xyz=3)")
for p, (x, y, z) in enumerate(orb_centers_mo):
    cl = next((i+1 for i, c in enumerate(clusters) if (p+1) in c), "?")
    print(f"  MO {p+1:2d} (C{cl}):  x={x:8.4f}  y={y:8.4f}  z={z:8.4f}")

# ---- Classical traceless dipole-dipole tensor between all orbital pairs ----
def orbital_dipolar_tensor(centers):
    n = len(centers)
    T = np.zeros((3, 3, n, n))
    for p in range(n):
        for q in range(n):
            if p == q:
                continue
            R  = centers[p] - centers[q]
            r2 = float(np.dot(R, R))
            if r2 < 1e-8:
                continue
            r5 = r2 ** 2.5
            for a in range(3):
                for b in range(3):
                    T[a, b, p, q] = (3 * R[a] * R[b] - (r2 if a == b else 0.0)) / r5
    return T

T_cl = orbital_dipolar_tensor(orb_centers_mo)   # (3, 3, n_act, n_act), Bohr^-3
print(f"\nClassical spin-dipolar tensor shape: {T_cl.shape}")
print(f"Max |T_cl|: {np.max(np.abs(T_cl)):.4e} Bohr^-3")

# ---- D_SS estimate for dominant C1-C3 Cr pair (orbital-center approx) ----
# For 26-orbital: K_IK_exch has been transposed → (norb_I, norb_I, norb_K, norb_K)
# K_IK_exch[p,r,q,s] = (pr|qs). Exchange diagonal: K_IK_exch[p,p,q,q] = (pp|qq) = J (Coulomb)
# Instead, recompute K_pq = h2[p, q, p, q] = (pq|pq) directly:
p_glob = list(range(sl_I.start, sl_I.stop))   # Cr1 global orbital indices
q_glob = list(range(sl_K.start, sl_K.stop))   # Cr3 global orbital indices

alpha_fs    = 1.0 / 137.035999084
g_e         = 2.0023
S_total_HS  = 3.0    # Cr1-Cr3 HS ground state: S = 3/2 + 3/2 = 3
prefac_DSS  = g_e**2 * alpha_fs**2 / 4.0 / (2 * S_total_HS * (2 * S_total_HS - 1))

D_SS_cl = np.zeros((3, 3))
for p in p_glob:
    for q in q_glob:
        K_pq     = h2[p, q, p, q]    # K_pq = (pq|pq) exchange integral
        D_SS_cl += K_pq * T_cl[:, :, p, q]

D_SS_cl    *= prefac_DSS
D_SS_cl_cm1 = D_SS_cl * 219474.63

ev_ss = np.linalg.eigvalsh(D_SS_cl_cm1)
D_val = ev_ss[2] - ev_ss[0]
E_val = (ev_ss[1] - ev_ss[0]) / 2
print(f"\nD_SS estimate (C1-C3, orbital-center approx, S={int(S_total_HS)}):")
print(f"  D = {D_val:8.2f} cm^-1")
print(f"  E = {E_val:8.2f} cm^-1")
print("  D_SS tensor (cm^-1):")
print(np.round(D_SS_cl_cm1, 4))
print("\nNote: point-dipole approx — exact D_SS needs spin-resolved 2-RDM (spin_correlators.jl)")

Orbital centers in active MO basis (Bohr):
  shape: (26, 3)  (n_act=26, xyz=3)
  MO  1 (C1):  x=  0.0059  y= -0.0089  z= -0.0064
  MO  2 (C1):  x=  0.0058  y= -0.0119  z= -0.0079
  MO  3 (C1):  x=  1.5325  y= -2.4194  z= -1.7908
  MO  4 (C1):  x=  1.5318  y= -2.4192  z= -1.7913
  MO  5 (C1):  x=  1.5621  y= -2.4655  z= -1.8237
  MO  6 (C1):  x=  1.5617  y= -2.4657  z= -1.8252
  MO  7 (C1):  x=  1.4978  y= -2.3644  z= -1.7494
  MO  8 (C1):  x=  1.5470  y= -2.4606  z= -1.8359
  MO  9 (C1):  x=  1.8011  y= -2.8458  z= -2.1139
  MO 10 (C1):  x=  1.8127  y= -2.8485  z= -2.0936
  MO 11 (C2):  x=  0.0002  y= -0.0015  z=  0.0001
  MO 12 (C2):  x= -0.0006  y=  0.0001  z=  0.0007
  MO 13 (C2):  x= -0.0000  y=  0.0000  z= -0.0000
  MO 14 (C2):  x= -0.0009  y= -0.0008  z=  0.0004
  MO 15 (C2):  x=  0.0003  y=  0.0005  z= -0.0007
  MO 16 (C2):  x=  0.0000  y=  0.0001  z= -0.0000
  MO 17 (C3):  x= -0.0080  y=  0.0127  z=  0.0087
  MO 18 (C3):  x= -0.0080  y=  0.0095  z=  0.0114
  MO 19 (C3):  x= -1.

## 5. Save All Integrals

In [28]:
# ---- Hamiltonian integrals (existing) ----
np.save("ints_h0_26", h0)
np.save("ints_h1_26", h1)
np.save("ints_h2_26", h2)
np.save("mo_coeffs_26", Cact)
np.save("overlap_mat_26", S)

# SCF 1-RDM projected into active space (needed for CMF initial guess)
np.save("Pa_26", Pa_act)
np.save("Pb_26", Pb_act)

# ---- Property integrals ----
np.save("dipole_ints_26", dip_mo)     # shape (3, n_act, n_act)  [x, y, z]
np.save("nabla_ints_26",  nabla_mo)   # shape (3, n_act, n_act)  [x, y, z], antisymmetric

print("Saved:")
print("  ints_h0_26.npy       scalar")
print(f"  ints_h1_26.npy       {h1.shape}")
print(f"  ints_h2_26.npy       {h2.shape}")
print(f"  mo_coeffs_26.npy     {Cact.shape}")
print(f"  Pa_26.npy            {Pa_act.shape}")
print(f"  Pb_26.npy            {Pb_act.shape}")
print(f"  dipole_ints_26.npy   {dip_mo.shape}")
print(f"  nabla_ints_26.npy    {nabla_mo.shape}")

# ---- SOC integrals ----
np.save("soc_Lx_26", soc_Lx)   # (n_act, n_act), imaginary part of <p|Lx|q>
np.save("soc_Ly_26", soc_Ly)
np.save("soc_Lz_26", soc_Lz)
print(f"  soc_Lx_26.npy        {soc_Lx.shape}")
print(f"  soc_Ly_26.npy        {soc_Ly.shape}")
print(f"  soc_Lz_26.npy        {soc_Lz.shape}")

# ---- Exchange integrals ----
np.save("K_12_exch_26", K_IJ_exch)   # (norb_I, norb_I, norb_J, norb_J): K[p,r,q,s]=(pr|qs)
np.save("K_13_exch_26", K_IK_exch)   # (norb_I, norb_I, norb_K, norb_K): K[p,r,q,s]=(pr|qs)
np.save("K_23_exch_26", K_JK_exch)   # (norb_J, norb_J, norb_K, norb_K): K[p,r,q,s]=(pr|qs)
print(f"  K_12_exch_26.npy     {K_IJ_exch.shape}  (pr|qs), p,r∈C1; q,s∈C2")
print(f"  K_13_exch_26.npy     {K_IK_exch.shape}  (pr|qs), p,r∈C1; q,s∈C3")
print(f"  K_23_exch_26.npy     {K_JK_exch.shape}  (pr|qs), p,r∈C2; q,s∈C3")

# ---- Orbital centers and classical spin-dipolar tensor ----
np.save("orb_centers_26", orb_centers_mo)  # (n_act, 3), Bohr
np.save("T_cl_SS_26",     T_cl)            # (3, 3, n_act, n_act), Bohr^-3
print(f"  orb_centers_26.npy   {orb_centers_mo.shape}  orbital centroids (Bohr)")
print(f"  T_cl_SS_26.npy       {T_cl.shape}  classical spin-dipolar tensor (Bohr^-3)")

Saved:
  ints_h0_26.npy       scalar
  ints_h1_26.npy       (26, 26)
  ints_h2_26.npy       (26, 26, 26, 26)
  mo_coeffs_26.npy     (366, 26)
  Pa_26.npy            (26, 26)
  Pb_26.npy            (26, 26)
  dipole_ints_26.npy   (3, 26, 26)
  nabla_ints_26.npy    (3, 26, 26)
  soc_Lx_26.npy        (26, 26)
  soc_Ly_26.npy        (26, 26)
  soc_Lz_26.npy        (26, 26)
  K_12_exch_26.npy     (10, 10, 6, 6)  (pr|qs), p,r∈C1; q,s∈C2
  K_13_exch_26.npy     (10, 10, 10, 10)  (pr|qs), p,r∈C1; q,s∈C3
  K_23_exch_26.npy     (6, 6, 10, 10)  (pr|qs), p,r∈C2; q,s∈C3
  orb_centers_26.npy   (26, 3)  orbital centroids (Bohr)
  T_cl_SS_26.npy       (3, 3, 26, 26)  classical spin-dipolar tensor (Bohr^-3)


## 6. What Goes Into the Julia CMF Script

After this notebook runs, `cmf.jl` must apply the CMF orbital rotation `U`
to the property integrals as well as the Hamiltonian. Add the block below
to `cmf.jl` immediately after `ints = orbital_rotation(ints, U)`:

```julia
# Rotate property integrals with the CMF unitary U
# h'[p,q] = U' * h[p,q] * U  (same transformation as ints.h1)
dip_mo  = npzread("dipole_ints.npy")   # shape [3, n_act, n_act] in Julia
dip_cmf = similar(dip_mo)
for x in 1:3
    dip_cmf[x,:,:] .= U' * dip_mo[x,:,:] * U
end

nabla_mo  = npzread("nabla_ints.npy")
nabla_cmf = similar(nabla_mo)
for x in 1:3
    nabla_cmf[x,:,:] .= U' * nabla_mo[x,:,:] * U
end

@save "data_cmf.jld2" clusters init_fspace ints d1 e_cmf U dip_cmf nabla_cmf
```

Then in the TPSCI script, after solving for the wavefunction:

```julia
@load "data_cmf.jld2" clusters init_fspace ints d1 dip_cmf nabla_cmf

# ... tpsci_ci(...) to get e0a, v0a ...

γ_aa, γ_bb = TPSChem.compute_1rdm(v0a, cluster_ops)

f = TPSChem.compute_oscillator_strengths(
        e0a, γ_aa, γ_bb,
        dip_cmf[1,:,:], dip_cmf[2,:,:], dip_cmf[3,:,:];
        ref_root = 1)

TPSChem.print_stick_spectrum(e0a, f; units=:ev)

ω, I = TPSChem.absorption_spectrum(e0a, f; σ=0.005, lineshape=:lorentzian)
```

## 7. Optional: Visualize Molecular Orbitals

In [29]:
# Write Molden files for visualization (e.g. with Avogadro or Jmol)
pyscf.tools.molden.from_mo(pymol, "Cact_26.molden",  Cact)
pyscf.tools.molden.from_mo(pymol, "Cenv_26.molden",  Cenv)

for i, Cf in enumerate(Cfrags):
    pyscf.tools.molden.from_mo(pymol, f"Cfrag{i}_26.molden", Cf)

print("Molden files written.")

Molden files written.


## Appendix: Integral Conventions

| File | Shape | Contents | Sign / notes |
|---|---|---|---|
| `ints_h0.npy` | scalar | nuclear repulsion + env. embedding energy | positive |
| `ints_h1.npy` | `(n,n)` | 1e Hamiltonian in active MO basis | includes $J_{\rm env} - \frac{1}{2}K_{\rm env}$ |
| `ints_h2.npy` | `(n,n,n,n)` | 2e repulsion integrals $(pq|rs)$ | chemist notation |
| `dipole_ints.npy` | `(3,n,n)` | $\langle p\|r_\alpha\|q\rangle$ in active MO | **positive** (position); multiply by $-e$ for dipole |
| `nabla_ints.npy` | `(3,n,n)` | $\langle p\|\nabla_\alpha\|q\rangle$ in active MO | antisymmetric; imaginary in real-orbital basis |

The **electronic dipole moment** is $\mu_\alpha = -\mathrm{Tr}(\mu^\alpha \cdot \gamma)$
where the minus sign accounts for the electron charge.  The nuclear contribution
$\mu_\alpha^{\rm nuc} = \sum_A Z_A R_{A\alpha}$ must be added separately.

The **oscillator strength** in the length gauge is
$$f_{0n} = \frac{2}{3}\,\Delta E_{0n}\,
\sum_{\alpha\in\{x,y,z\}} |\mathrm{Tr}(\mu^\alpha \cdot \gamma^{0n})|^2$$
which is what `TPSChem.compute_oscillator_strengths` computes directly from
`dipole_ints.npy` (no sign flip needed; the sign cancels in $|\cdot|^2$).